# MOMENT cls: Temporal (training-dynamics) Attack (gray-box) (TV4)

Chạy **sau** `moment_colab.ipynb` (cần `ckpt/foundation_cls/full_<level>_s42.done.json` và `full_<level>_s42_logits_ep{1,2,5,10}.pt`). Notebook này **không cần GPU** và không cần weight của model.

Với cls không có chuỗi residual để lấy DFT như fcst, nên "temporal" ở đây là **thời gian huấn luyện**: attacker nhìn **quỹ đạo** của mỗi bản ghi qua các checkpoint epoch 1, 2, 5, 10 thay vì chỉ epoch cuối. Bản ghi member thường bị "học thuộc" sớm và ổn định (loss giảm nhanh, margin lớn dần), còn non-member thì không.

Đặc trưng mỗi bản ghi (3 × 4 epoch + 5 = 17 chiều):
1. Mỗi epoch: CE, entropy, margin (logit đúng − logit lớn nhất trong các lớp còn lại).
2. Động học: chênh CE giữa các epoch liên tiếp (3), độ dốc CE theo epoch (1), độ dốc margin theo epoch (1).

**Threat model mạnh hơn Loss/Conf/Entropy**: attacker truy cập được các checkpoint trung gian (ví dụ model được release nhiều phiên bản) và nhãn của bản ghi cần kiểm tra, cộng một ít nhãn member/non-member để huấn luyện bộ phân loại (cross-fitting 5-fold: mỗi bản ghi được chấm bởi bộ phân loại chưa từng thấy nó). Phải nêu rõ điều này khi so sánh với Loss.

Ablation đi kèm: classifier tuyến tính cùng đặc trưng; chỉ đặc trưng epoch cuối qua cùng pipeline (cho thấy quỹ đạo thêm được bao nhiêu); chỉ CE epoch cuối qua cùng pipeline.

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'   # must be set before torch touches CUDA
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/Final_Project'
MOMENT_SRC = f'{DRIVE_ROOT}/moment'          # folder containing setup.py
DATASET_DIR = f'{DRIVE_ROOT}/dataset'        # shared split script + splits/
PTBXL_ZIP = f'{DATASET_DIR}/archive (3).zip'
RESULTS_DIR = f'{DRIVE_ROOT}/results/foundation/cls'

In [ ]:
# Copy source to local disk (faster than installing from Drive), then install
!rm -rf /content/moment && cp -r {MOMENT_SRC} /content/moment
!pip install -q /content/moment wfdb scikit-learn pandas tqdm

## 1. Cấu hình

In [ ]:
SEED = 42
LEVEL = 'record'            # record-level split, must match the cls run
MODEL_NAME = 'AutonLab/MOMENT-1-large'
MODE = 'full'
EVAL_AT = [1, 2, 5, 10]     # epochs saved by moment_colab (logits_ep{n}.pt); all must exist

N_VAL = 500
N_BOOT = 1000
N_FOLDS = 5                 # cross-fitting folds for the attack classifier
CKPT_DIR = f'{DRIVE_ROOT}/ckpt/foundation_cls'
TAG = f'{MODE}_{LEVEL}_s{SEED}'

PTBXL_ROOT = '/content/ptbxl_raw'
SPLIT_PATH = f'{DATASET_DIR}/splits/ptbxl_{LEVEL}_s{SEED}.json'
CACHE_DIR = f'/content/cache/{LEVEL}_s{SEED}'

## 2. Dữ liệu PTB-XL
Giải nén `archive (3).zip` từ Drive vào disk local của Colab (đọc nhanh hơn Drive).

In [ ]:
import os, glob
if not glob.glob(f'{PTBXL_ROOT}/**/ptbxl_database.csv', recursive=True):
    os.makedirs(PTBXL_ROOT, exist_ok=True)
    assert os.path.exists(PTBXL_ZIP), f'missing {PTBXL_ZIP}; {DATASET_DIR} has {os.listdir(DATASET_DIR)}'
    !unzip -q -o "{PTBXL_ZIP}" -d {PTBXL_ROOT}
found = glob.glob(f'{PTBXL_ROOT}/**/ptbxl_database.csv', recursive=True)
assert found, f'no ptbxl_database.csv after unzip; {PTBXL_ROOT} has {os.listdir(PTBXL_ROOT)}'
PTBXL_DIR = os.path.dirname(found[0])
print(PTBXL_DIR, os.listdir(PTBXL_DIR))

In [ ]:
import json, random, subprocess, sys
import numpy as np, torch, pandas as pd
from momentfm.data.ptbxl_classification_dataset import PTBXL_dataset

def control_randomness(seed=SEED):
    random.seed(seed); os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
control_randomness()

# shared split from dataset/make_split.py (generated here if missing)
if not os.path.exists(SPLIT_PATH):
    subprocess.run([sys.executable, f'{DATASET_DIR}/make_split.py', '--ptbxl-dir', PTBXL_DIR,
                    '--level', LEVEL, '--seed', str(SEED), '--n-val', str(N_VAL)], check=True)
split = json.load(open(SPLIT_PATH))
meta = json.load(open(SPLIT_PATH.replace('.json', '.meta.json')))
print({k: len(v) for k, v in split.items()}, 'sha256', meta['sha256'][:12])
if len(split['test']) < 10000:
    print('WARNING: fewer than 10k non-members; TPR@0.1%FPR rests on', int(0.001 * len(split['test'])), 'samples')

class Config:
    base_path = PTBXL_DIR
    cache_dir = CACHE_DIR
    path_to_pids = SPLIT_PATH
    load_cache = True
    fs = 100
    output_type = 'single'    # lowercase: modify_df compares against 'single'
    seq_len = 512
os.makedirs(CACHE_DIR, exist_ok=True)

member_ds = PTBXL_dataset(Config, phase='train')
val_ds    = PTBXL_dataset(Config, phase='val')
nonmem_ds = PTBXL_dataset(Config, phase='test')
print(len(member_ds), len(val_ds), len(nonmem_ds), member_ds.timeseries.shape)

## 2. Quỹ đạo logit qua các epoch

In [ ]:
import torch
import torch.nn.functional as F

done = json.load(open(f'{CKPT_DIR}/{TAG}.done.json'))          # fails loudly if cls training never finished
lg = {ep: torch.load(f'{CKPT_DIR}/{TAG}_logits_ep{ep}.pt') for ep in EVAL_AT}
y_m, y_n = np.asarray(member_ds.labels).astype(int), np.asarray(nonmem_ds.labels).astype(int)
assert all(len(lg[ep]['lm']) == len(y_m) and len(lg[ep]['ln']) == len(y_n) for ep in EVAL_AT)

def per_epoch(logits, labels):
    # CE, entropy, margin for one epoch -> three arrays of shape [N]
    logp = F.log_softmax(logits.float(), -1)
    lab = torch.as_tensor(labels)[:, None]
    ce = -logp.gather(1, lab).squeeze(1)
    ent = -(logp.exp() * logp).sum(1)
    true = logits.float().gather(1, lab).squeeze(1)
    other = logits.float().scatter(1, lab, float('-inf')).max(1).values
    return ce.numpy(), ent.numpy(), (true - other).numpy()

def slope(v, t):
    # least-squares slope of each row of v [N, T] against epochs t [T]
    t = np.asarray(t, dtype=np.float64); t = t - t.mean()
    return (v * t).sum(1) / (t ** 2).sum()

def trajectory_features(key, labels):
    ce, ent, mar = (np.stack(z, 1) for z in zip(*[per_epoch(lg[ep][key], labels) for ep in EVAL_AT]))   # each [N, n_epochs]
    dce = np.diff(ce, axis=1)                                   # CE change between consecutive checkpoints
    dyn = np.stack([slope(ce, EVAL_AT), slope(mar, EVAL_AT)], 1)
    final = np.stack([ce[:, -1], ent[:, -1], mar[:, -1]], 1)    # what a single-checkpoint attacker sees
    return np.concatenate([ce, ent, mar, dce, dyn], 1), final, ce[:, -1]

Xm, Fm, Lm = trajectory_features('lm', y_m)
Xn, Fn, Ln = trajectory_features('ln', y_n)
X = np.r_[Xm, Xn]; Xf = np.r_[Fm, Fn]; y = np.r_[np.ones(len(Xm)), np.zeros(len(Xn))]
loss_all = -np.r_[Lm, Ln]                                        # raw Loss attack score at the last epoch
print('features', X.shape, 'final-epoch features', Xf.shape, 'members', len(Xm), 'non-members', len(Xn))

## 3. Tấn công (cross-fitting) + ablation

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve, auc as sk_auc

def tpr_at(fpr, tpr, t):
    m = fpr <= t
    return float(tpr[m].max()) if m.any() else 0.0

def mia_metrics(sm, sn):
    fpr, tpr, _ = roc_curve(np.r_[np.ones(len(sm)), np.zeros(len(sn))], np.r_[sm, sn])
    return tpr_at(fpr, tpr, 1e-3), tpr_at(fpr, tpr, 1e-2), float(sk_auc(fpr, tpr))

def bootstrap_ci(sm, sn, n=N_BOOT, seed=SEED):
    rng = np.random.default_rng(seed)
    res = np.array([mia_metrics(sm[rng.integers(0, len(sm), len(sm))],
                                sn[rng.integers(0, len(sn), len(sn))])[:2] for _ in range(n)])
    lo, hi = np.percentile(res, [2.5, 97.5], axis=0)
    return [float(lo[0]), float(hi[0])], [float(lo[1]), float(hi[1])]

def cross_fit(make_clf, X, y):
    # every sample is scored by a classifier that never saw it, so scores of members and non-members are comparable
    score = np.zeros(len(y))
    for tr, te in StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(X, y):
        clf = make_clf().fit(X[tr], y[tr])
        score[te] = clf.predict_proba(X[te])[:, 1]
    return score

hgb = lambda: HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, random_state=SEED)
makers = {
    # main attack, fixed in advance: gradient boosting on the whole training trajectory
    'temporal':       (hgb, X),
    # ablation 1: same features, linear classifier
    'temporal_lr':    (lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)), X),
    # ablation 2: same classifier, final-epoch features only (CE, entropy, margin): what the trajectory adds
    'final_epoch_clf': (hgb, Xf),
    # ablation 3: same pipeline with only the final-epoch CE
    'loss_only_clf':  (hgb, loss_all[:, None]),
}
S, M = {}, {}
nm = len(Xm)
for name, (mk, feats) in makers.items():
    S[name] = cross_fit(mk, feats, y)
    M[name] = mia_metrics(S[name][:nm], S[name][nm:])
M['loss (raw -CE)'] = mia_metrics(loss_all[:nm], loss_all[nm:])
print(pd.DataFrame(M, index=['TPR@0.1%', 'TPR@1%', 'AUC']).T.round(4))
if len(Xn) < 10000:
    print('WARNING: fewer than 10k non-members; TPR@0.1%FPR rests on', int(0.001 * len(Xn)), 'samples')

## 4. Ghi kết quả

In [ ]:
sm, sn = S['temporal'][:nm], S['temporal'][nm:]
c01, c1 = bootstrap_ci(sm, sn)
t01, t1, au = M['temporal']
hist = [r for r in done.get('history', []) if r.get('epoch') == max(EVAL_AT)]
hist = hist[-1] if hist else {}
out = {
    'model': 'foundation', 'task': 'cls', 'attack': 'temporal',
    'owner_tv': 4, 'level': LEVEL, 'dataset': 'ptbxl', 'seed': SEED,
    'n_member': int(len(sm)), 'n_nonmember': int(len(sn)),
    'scores_member': np.round(sm, 6).tolist(), 'scores_nonmember': np.round(sn, 6).tolist(),
    'tpr_at_0.1fpr': t01, 'tpr_at_1fpr': t1, 'auc': au,
    'bootstrap_ci': {'n_resamples': N_BOOT, 'confidence': 0.95, 'tpr_at_0.1fpr': c01, 'tpr_at_1fpr': c1},
    'utility': {k: hist[k] for k in ('train_acc', 'test_acc', 'gap') if k in hist},
    'n_shadow': None,
    'extra': {'tuning': 'fixed recipe, no search', 'mode': MODE, 'eval_epochs': EVAL_AT, 'backbone': MODEL_NAME, 'variant': 'training-dynamics',
              'classifier': 'HistGradientBoosting', 'n_folds': N_FOLDS,
              'features': 'per-epoch CE/entropy/margin + consecutive CE change + CE and margin slope over checkpoints',
              'threat_model': 'needs intermediate checkpoints, true labels and a labelled member/non-member subset',
              'ablation': {k: {'tpr_at_0.1fpr': v[0], 'tpr_at_1fpr': v[1], 'auc': v[2]} for k, v in M.items()}},
}
os.makedirs(RESULTS_DIR, exist_ok=True)
fn = f'{RESULTS_DIR}/temporal.json'
json.dump(out, open(fn, 'w'))
print('saved', fn, '| TPR@0.1%', round(t01, 4), c01, '| TPR@1%', round(t1, 4), c1, '| AUC', round(au, 4))

## 5. ROC log–log

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(5, 5))
for name, s in {'temporal (trajectory)': S['temporal'], 'temporal LR': S['temporal_lr'], 'final epoch + clf': S['final_epoch_clf'],
                'loss only + clf': S['loss_only_clf'], 'raw loss': loss_all}.items():
    fpr, tpr, _ = roc_curve(y, s)
    ax.plot(fpr, tpr, label=name)
ax.plot([1e-4, 1], [1e-4, 1], 'k--', lw=0.8)
ax.set_xscale('log'); ax.set_yscale('log'); ax.set_xlim(1e-4, 1); ax.set_ylim(1e-4, 1)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR'); ax.legend(fontsize=8); ax.set_title(f'MOMENT PTB-XL cls, {LEVEL}-level, Temporal attack')
plt.show()